In [ ]:
import dynamiqs as dq
dq.set_precision('double')
import numpy as np
from scipy.sparse.linalg import eigs
import scipy.sparse as sp
import scipy.sparse.linalg as spla

import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting

In [ ]:
# --- constants ---
ħ = 1
e = 1
phi_quantum = 1 / 2

w_a = 25.338776456203686
kappa_b = 5 / 10.4*2
E_J = 37*2*np.pi /2
phi_a = 0.11
phi_b = 0.204
epsilon_p = 0.02
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b 
kappa_2 = 4*g**2/kappa_b
kappa_1 = 0.005*kappa_2



In [ ]:
n_a = 20
n_b = 11
N = n_a * n_b

# Operators
a = dq.destroy(n_a)
b = dq.destroy(n_b)
Ia = dq.eye(n_a)
Ib = dq.eye(n_b)

a_tot = dq.tensor(a, Ib)
b_tot = dq.tensor(Ia, b)

# Correct nonlinear term
term = dq.tensor(a @ a, dq.dag(b))
non_linear_op = term + dq.dag(term)
drive_op = b_tot + b_tot.dag()

La = np.sqrt(kappa_1) * a_tot
Lb = np.sqrt(kappa_b) * b_tot

# Convert to sparse CSC
La = sp.csc_matrix(La.to_numpy().astype(np.complex128))
Lb = sp.csc_matrix(Lb.to_numpy().astype(np.complex128))
I = sp.identity(N, dtype=np.complex128, format="csc")

def dissipator(L):
    LdL = L.getH() @ L
    return (
        sp.kron(L.conjugate(), L, format="csc")
        - 0.5 * sp.kron(I, LdL, format="csc")
        - 0.5 * sp.kron(LdL.T, I, format="csc")
    )



In [ ]:
alpha_sq = 2
epsilon_d = 2*alpha_sq*g
H = g * non_linear_op + (epsilon_d / 2) * drive_op
H = sp.csc_matrix(H.to_numpy().astype(np.complex128))

Lio = (
-1j * (sp.kron(I, H, format="csc") - sp.kron(H.T, I, format="csc"))
+ dissipator(La)
+ dissipator(Lb)
)


In [ ]:
def identity_vec(N):
    return np.eye(N, dtype=np.complex128).reshape(N*N, order="F")

def project_trace_zero(x, N):
    Ivec = identity_vec(N)
    tr = np.vdot(Ivec, x)   # = Tr(rho)
    return x - (tr / N) * Ivec

def normalize(x):
    nrm = np.linalg.norm(x)
    if nrm == 0:
        raise ValueError("Zero vector encountered.")
    return x / nrm

In [ ]:
lu = spla.splu(Lio)

In [ ]:
dim = N * N
Ibig = sp.identity(dim, dtype=np.complex128, format="csc")

# Factorize L - sigma I
rng = np.random.default_rng(None)
x = rng.normal(size=dim) + 1j * rng.normal(size=dim)
x = project_trace_zero(x, N)
x = normalize(x)

eig_old = None
eig_record=[]
num_iter = 5
for k in range(num_iter):
    print(k)
    y = lu.solve(x)

    # remove trace component
    y = project_trace_zero(y, N)
    x = normalize(y)

    # estimate eigenvalue of original Lindbladian
    eig = np.vdot(x, Lio @ x) / np.vdot(x, x)

    eig_record.append(eig.real)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(-np.array(eig_record))
plt.yscale("log")
plt.ylim(1e-9,1.02e-9)
plt.show()

In [ ]:
n_a = 25

# --- operators ---
a = dq.destroy(n_a)
Ia = dq.eye(n_a)

# pure dissipative cat stabilization
H = 0 * Ia
jump_op_1 = np.sqrt(kappa_1)*a

# --- convert to numpy ---
I_np = np.eye(n_a, dtype=np.complex128)
L_np_1 = jump_op_1.to_numpy()
H_np = H.to_numpy()

LdL_1 = L_np_1.conj().T @ L_np_1

jump_op_2 = np.sqrt(kappa_2) * (a @ a - alpha_sq * Ia)

# --- convert to numpy ---
L_np_2 = jump_op_2.to_numpy()
LdL_2 = L_np_2.conj().T @ L_np_2

# --- Liouvillian in vectorized form (column stacking) ---
liouvillian = (
    -1j * (np.kron(I_np, H_np) - np.kron(H_np.T, I_np))
    + np.kron(L_np_2.conj(), L_np_2)
    - 0.5 * np.kron(I_np, LdL_2)
    - 0.5 * np.kron(LdL_2.T, I_np)
    + np.kron(L_np_1.conj(), L_np_1)
    - 0.5 * np.kron(I_np, LdL_1)
    - 0.5 * np.kron(LdL_1.T, I_np)
)

vals = np.linalg.eigvals(liouvillian).real
print(-np.sort(vals)[-1])


In [ ]:
np.sort(vals)[-10:]

In [ ]:
vals, vecs = np.linalg.eig(liouvillian)
vec_zero = vecs[:,np.argsort(vals)[-1]]
vec_1 = vecs[:,np.argsort(vals)[-2]]

In [ ]:
vec_1

In [ ]:
import json
with open("/Users/tmalas/Desktop/Dev/Simulations_local/Output/power_iteration_utils/vec_1_n_a_25_30_04_2026.json", "w") as f:
    json.dump(vec_1.real.tolist(), f)

In [ ]:
rng = np.random.default_rng(None)

x = rng.normal(size=n_a*n_a) + 1j * rng.normal(size=n_a*n_a)
x = project_trace_zero(x, n_a)
x = normalize(x)

eig_record = []
res_record = []
num_iter = 60000
dt = 1
liouvillian_dt = dt*liouvillian

for k in range(num_iter):
    if k%1000 ==0 : print(k)
    y = spla.expm_multiply(liouvillian_dt, x)
    x = normalize(y)

    eig = np.vdot(x, liouvillian @ x) / np.vdot(x, x)
    res = np.linalg.norm(liouvillian @ x - eig * x)

    eig_record.append(eig.real)
    res_record.append(res)

In [ ]:
plt.plot(np.abs(np.array(res_record)))
plt.yscale("log")
plt.show()

In [ ]:
plt.plot(np.abs(np.array(eig_record)))
plt.yscale("log")
plt.show()

In [ ]:
eig_record[-1]

In [ ]:
rng = np.random.default_rng(None)

x = rng.normal(size=n_a*n_a) + 1j * rng.normal(size=n_a*n_a)
x = project_trace_zero(x, n_a)
x = normalize(x)

eig_record = []
res_record = []
num_iter = 10
dt = 1e6
liouvillian_dt = dt*liouvillian

for k in range(num_iter):
    print(k)
    y = spla.expm_multiply(liouvillian_dt, x)
    x = normalize(y)

    eig = np.vdot(x, liouvillian @ x) / np.vdot(x, x)
    res = np.linalg.norm(liouvillian @ x - eig * x)

    eig_record.append(eig.real)
    res_record.append(res)

In [ ]:
plt.plot(-np.array(eig_record))
plt.yscale("log")
plt.show()

In [ ]:
eig_record[-1]

# We try to guide the search with a good choice of initial vector

In [ ]:
eig_record = []
res_record = []
num_iter = 20000
dt = 1
liouvillian_dt = dt*liouvillian

x = vec_1
for k in range(num_iter):
    if k%1000 ==0 : print(k)
    y = spla.expm_multiply(liouvillian_dt, x)
    x = normalize(y)

    eig = np.vdot(x, liouvillian @ x) / np.vdot(x, x)
    res = np.linalg.norm(liouvillian @ x - eig * x)

    eig_record.append(eig.real)
    res_record.append(res)

In [ ]:
plt.plot(np.abs(np.array(eig_record)))
plt.yscale("log")
plt.show()

In [ ]:
def mat_to_vec(rho_mat):
    return rho_mat.reshape(-1, order='F')

def vec_to_mat(rho_vec, N):
    return rho_vec.reshape((N, N), order='F')
rho_1 = dq.asqarray(vec_to_mat(vec_1, n_a))
rho_2 = dq.tensor(rho_1,dq.coherent(n_b,0))
vec_2 = mat_to_vec(rho_2.to_numpy())

In [ ]:
rng = np.random.default_rng(None)

x = vec_2
x = normalize(x)

eig_record = []
res_record = []
num_iter = 200
dt = 1
Lio_dt = dt*Lio

for k in range(num_iter):
    print(k)
    y = spla.expm_multiply(Lio_dt, x)
    x = normalize(y)

    eig = np.vdot(x, liouvillian @ x) / np.vdot(x, x)
    res = np.linalg.norm(liouvillian @ x - eig * x)

    eig_record.append(eig.real)
    res_record.append(res)

In [ ]:
plt.plot(-np.array(eig_record))
#plt.plot((np.array(eig_record_im)))
plt.yscale("log")
plt.show()